# NYISO Behind-The-Meter Solar — Data Validation & Scenario Generation

This notebook validates the NYISO BTM solar data (estimated actuals + day-ahead zonal forecasts)
downloaded from `mis.nyiso.com` (P-70A / P-70B), and runs PGScen GEMINI scenario generation.

**Contents:**
1. Loading and exploring BTM solar data
2. Annual growth and installed capacity trends
3. Diurnal profiles by zone and season
4. Forecast accuracy (actual vs day-ahead forecast)
5. PGScen GEMINI scenario generation
6. Scenario visualization (20 days across 2024)
7. Calibration summary

**Prerequisite:** Run `scripts/07_download_nyiso_btm_solar.py` first (or data already in `data/NYISO_real/`).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Resolve PGSCEN_DIR whether the notebook is launched from the repo root or from notebooks/.
_CWD = Path.cwd()
PGSCEN_DIR = None
for _p in [_CWD, *_CWD.parents]:
    if _p.name == 'PGscen-2nd':
        PGSCEN_DIR = _p
        break
    if (_p / 'PGscen-2nd').is_dir():
        PGSCEN_DIR = _p / 'PGscen-2nd'
        break
if PGSCEN_DIR is None:
    raise RuntimeError(f'Could not locate PGscen-2nd from {_CWD}')
ROOT = PGSCEN_DIR.parent

if str(PGSCEN_DIR) not in sys.path:
    sys.path.insert(0, str(PGSCEN_DIR))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

plt.rcParams.update({
    'figure.figsize': (16, 6),
    'figure.dpi': 120,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

print(f'ROOT       = {ROOT}')
print(f'PGSCEN_DIR = {PGSCEN_DIR}')
print('Imports OK')

---
## 1. Loading BTM Solar Data

The data comes from NYISO OASIS:
- **Estimated Actuals** (P-70A): hourly BTM solar generation by zone, estimated from sampled installations
- **Day-Ahead Forecast** (P-70B): NYISO's day-ahead BTM solar forecast by zone

Available from November 2020 onward. Zones match the load data (11 zones + NYCA system total).

In [ ]:
import importlib.util

# The loader lives in PGscen-2nd/scripts/07_download_nyiso_btm_solar.py.
# Filenames starting with a digit aren't valid module names, so load by path.
_SCRIPT = PGSCEN_DIR / 'scripts' / '07_download_nyiso_btm_solar.py'
_spec = importlib.util.spec_from_file_location('btm_solar_loader', _SCRIPT)
_btm_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_btm_mod)
load_real_ny_btm_solar_data = _btm_mod.load_real_ny_btm_solar_data

DATA_DIR = './data/NYISO_real'

# Try PGscen-2nd first, then PGscen-main
for d in [PGSCEN_DIR / 'data' / 'NYISO_real', ROOT / 'PGscen-main' / 'data' / 'NYISO_real', Path(DATA_DIR)]:
    if list(d.glob('btm_solar_actual_*')):
        DATA_DIR = str(d)
        break

btm_actual, btm_forecast = load_real_ny_btm_solar_data(DATA_DIR)

# Add timezone info to actuals index for consistency
if btm_actual.index.tz is None:
    btm_actual.index = btm_actual.index.tz_localize('UTC')

ZONES = [c for c in btm_actual.columns if c != 'NYCA']

print(f'\n=== BTM Solar Estimated Actuals ===')
print(f'  Shape: {btm_actual.shape}')
print(f'  Period: {btm_actual.index.min()} -> {btm_actual.index.max()}')
print(f'  Zones: {list(btm_actual.columns)}')

print(f'\n=== BTM Solar Day-Ahead Forecasts ===')
print(f'  Shape: {btm_forecast.shape}')
print(f'  Period: {btm_forecast["Forecast_time"].min()} -> {btm_forecast["Forecast_time"].max()}')
print(f'  Issue_times: {btm_forecast["Issue_time"].nunique()}')

In [ ]:
# Quick look at the data
print('=== Actuals (first rows) ===')
display(btm_actual.head(10))
print('\n=== Actuals (stats) ===')
display(btm_actual.describe().round(1))

---
## 2. Annual Growth & Peak BTM Solar

NYISO BTM solar has been growing rapidly. We look at:
- Annual peak system MW
- Annual energy (GWh) by zone
- Monthly peak trends

In [ ]:
# Annual summary
annual = btm_actual.groupby(btm_actual.index.year).agg(['mean', 'max', 'sum'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Peak system MW by year
ax = axes[0]
peak_by_year = btm_actual['NYCA'].groupby(btm_actual.index.year).max()
ax.bar(peak_by_year.index, peak_by_year.values, color='gold', edgecolor='darkorange')
for x, y in zip(peak_by_year.index, peak_by_year.values):
    ax.text(x, y + 30, f'{y:.0f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Peak MW')
ax.set_title('NYCA BTM Solar Peak (MW)')

# Annual energy by zone
ax = axes[1]
energy_by_year = btm_actual[ZONES].groupby(btm_actual.index.year).sum() / 1000  # GWh
energy_by_year.plot(kind='bar', stacked=True, ax=ax, colormap='Spectral', edgecolor='none')
ax.set_xlabel('Year'); ax.set_ylabel('Energy (GWh)')
ax.set_title('BTM Solar Energy by Zone')
ax.legend(fontsize=7, ncol=3, loc='upper left')

# Monthly peak NYCA
ax = axes[2]
monthly_peak = btm_actual['NYCA'].groupby([btm_actual.index.year, btm_actual.index.month]).max()
monthly_peak.index = pd.to_datetime([f'{y}-{m:02d}-01' for y, m in monthly_peak.index])
ax.plot(monthly_peak.index, monthly_peak.values, 'o-', color='darkorange', markersize=3)
ax.set_xlabel('Date'); ax.set_ylabel('Peak MW')
ax.set_title('Monthly Peak BTM Solar (NYCA)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.suptitle('NYISO Behind-The-Meter Solar Growth', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\nAnnual peak NYCA (MW):')
print(peak_by_year.to_string())

---
## 3. Diurnal Profiles by Zone and Season

BTM solar follows a clean bell curve peaking around 16-18 UTC (12-2 PM Eastern).
Summer peaks are higher and wider; winter peaks are lower and narrower.

In [ ]:
# Diurnal profiles by season (using 2024 for most recent full year)
btm_2024 = btm_actual[btm_actual.index.year == 2024].copy()
btm_2024['hour'] = btm_2024.index.hour
btm_2024['month'] = btm_2024.index.month
btm_2024['season'] = btm_2024['month'].map(
    {12:'Winter', 1:'Winter', 2:'Winter',
     3:'Spring', 4:'Spring', 5:'Spring',
     6:'Summer', 7:'Summer', 8:'Summer',
     9:'Fall', 10:'Fall', 11:'Fall'}
)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
season_colors = {'Winter': 'steelblue', 'Spring': 'green', 'Summer': 'red', 'Fall': 'orange'}

for idx, zone in enumerate(ZONES):
    ax = axes.flatten()[idx]
    for season in ['Winter', 'Spring', 'Summer', 'Fall']:
        mask = btm_2024['season'] == season
        profile = btm_2024.loc[mask].groupby('hour')[zone].mean()
        ax.plot(profile.index, profile.values, color=season_colors[season],
                linewidth=2, label=season)
    ax.set_title(zone, fontsize=11)
    ax.set_xlabel('Hour (UTC)')
    ax.set_ylabel('MW')
    ax.set_xlim(0, 23)
    if idx == 0:
        ax.legend(fontsize=8)

# Last subplot: NYCA total
ax = axes.flatten()[11]
for season in ['Winter', 'Spring', 'Summer', 'Fall']:
    mask = btm_2024['season'] == season
    profile = btm_2024.loc[mask].groupby('hour')['NYCA'].mean()
    ax.plot(profile.index, profile.values, color=season_colors[season],
            linewidth=2, label=season)
ax.set_title('NYCA (System)', fontsize=11)
ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('MW')
ax.set_xlim(0, 23)
ax.legend(fontsize=8)

plt.suptitle('BTM Solar Diurnal Profiles by Zone and Season (2024, UTC)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: average hourly BTM solar by month (NYCA)
btm_all = btm_actual.copy()
btm_all['hour'] = btm_all.index.hour
btm_all['month'] = btm_all.index.month

# Use 2024 data
btm_2024h = btm_all[btm_all.index.year == 2024]
heatmap_data = btm_2024h.pivot_table(index='hour', columns='month',
                                      values='NYCA', aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax, annot=True, fmt='.0f',
            xticklabels=['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_xlabel('Month'); ax.set_ylabel('Hour (UTC)')
ax.set_title('Average BTM Solar Generation (MW) by Hour and Month (NYCA, 2024)')
plt.tight_layout()
plt.show()

---
## 4. Forecast Accuracy: Actual vs Day-Ahead Forecast

We compare the NYISO day-ahead BTM solar forecast against estimated actuals.
This is the deviation distribution that PGScen GEMINI will model.

In [ ]:
# Align actuals and forecasts
fc = btm_forecast.set_index('Forecast_time').drop(columns='Issue_time')
fc.index = pd.to_datetime(fc.index, utc=True)
common_idx = btm_actual.index.intersection(fc.index)

print(f'Common timestamps: {len(common_idx)}')
print(f'Period: {common_idx.min()} -> {common_idx.max()}')

# Deviations (actual - forecast)
dev = btm_actual.loc[common_idx] - fc.loc[common_idx]

# Only look at daytime hours (11-22 UTC ~ 6am-5pm ET)
DAYTIME_HOURS = list(range(11, 23))
daytime_mask = common_idx.hour.isin(DAYTIME_HOURS)
dev_day = dev.loc[daytime_mask]

print(f'\nDaytime hours: {len(dev_day)}')

In [ ]:
# Deviation distributions by zone (daytime only)
n_zones = len(ZONES) + 1  # +1 for NYCA
n_cols = 4
n_rows = (n_zones + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = np.atleast_2d(axes)

all_zones = ZONES + ['NYCA']
for idx, zone in enumerate(all_zones):
    row, col = idx // n_cols, idx % n_cols
    ax = axes[row, col]
    d = dev_day[zone].dropna()
    ax.hist(d, bins=80, color='gold' if zone != 'NYCA' else 'darkorange',
            alpha=0.7, edgecolor='none')
    ax.axvline(0, color='red', ls='--', linewidth=1)
    ax.set_title(f'{zone}\nmean={d.mean():.1f}, std={d.std():.1f} MW', fontsize=10)
    ax.set_xlabel('Actual - Forecast (MW)')

for idx in range(n_zones, n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.suptitle('BTM Solar Deviation Distribution by Zone (daytime only, 2021-2025)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Q-Q plots of deviations (daytime only) — check for heavy tails
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = np.atleast_2d(axes)

for idx, zone in enumerate(all_zones):
    row, col = idx // n_cols, idx % n_cols
    ax = axes[row, col]
    d = dev_day[zone].dropna().values
    stats.probplot(d, dist='norm', plot=ax)
    ax.set_title(zone, fontsize=11)
    ax.get_lines()[0].set_markersize(2)

for idx in range(n_zones, n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.suptitle('Normal Q-Q Plots of BTM Solar Deviations (daytime)\n'
             '(heavy tails justify the use of the GPD)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Forecast accuracy metrics by zone
metrics = []
for zone in all_zones:
    d = dev_day[zone].dropna()
    a = btm_actual.loc[dev_day.index, zone]
    f = fc.loc[dev_day.index, zone]
    mae = d.abs().mean()
    rmse = np.sqrt((d**2).mean())
    bias = d.mean()
    # nMAE as % of mean actual (daytime)
    nmae = mae / a.mean() * 100 if a.mean() > 0 else np.nan
    metrics.append({'Zone': zone, 'MAE (MW)': mae, 'RMSE (MW)': rmse,
                    'Bias (MW)': bias, 'nMAE (%)': nmae,
                    'Mean Actual (MW)': a.mean()})

metrics_df = pd.DataFrame(metrics).set_index('Zone').round(2)
display(metrics_df)

In [ ]:
# 20-day time series: actual vs forecast (NYCA)
# Use the Eastern-day window (same as forecast blocks and scenario generation)
dates_2024 = [
    '2024-01-22', '2024-02-14', '2024-03-10', '2024-03-25', '2024-04-15',
    '2024-05-05', '2024-05-20', '2024-06-10', '2024-06-21', '2024-07-04',
    '2024-07-20', '2024-08-05', '2024-08-25', '2024-09-10', '2024-09-28',
    '2024-10-15', '2024-10-30', '2024-11-15', '2024-12-10', '2024-12-25',
]

fig, axes = plt.subplots(10, 2, figsize=(18, 40), sharex=True)
for i, date in enumerate(dates_2024):
    ax = axes.flatten()[i]
    
    # Find the forecast block for this day (same logic as run_btm_scenarios)
    target = pd.Timestamp(date, tz='UTC')
    fc_day = btm_forecast[
        (btm_forecast['Forecast_time'] >= target) &
        (btm_forecast['Forecast_time'] < target + pd.Timedelta(hours=30))
    ]
    if len(fc_day) == 0:
        ax.set_title(f'{date} - no data'); continue
    
    it0 = fc_day['Issue_time'].iloc[0]
    block = btm_forecast[btm_forecast['Issue_time'] == it0].sort_values('Forecast_time')
    ts_idx = pd.DatetimeIndex(block['Forecast_time'].values).tz_localize('UTC')
    
    # Align tz between actual index and forecast timestamps
    if btm_actual.index.tz is None:
        ts_lookup = ts_idx.tz_localize(None)
    else:
        ts_lookup = ts_idx
    
    act = btm_actual.loc[ts_lookup, 'NYCA'].values
    fc_vals = block.set_index('Forecast_time').sort_index()['NYCA'].values
    
    ax.fill_between(range(len(act)), 0, act, alpha=0.2, color='gold')
    ax.plot(range(len(act)), act, color='red', linewidth=1.5,
            marker='o', ms=2, label='Actual')
    ax.plot(range(len(fc_vals)), fc_vals, color='blue', linewidth=1.5,
            ls='--', label='DA Forecast')
    mae = np.abs(act - fc_vals).mean()
    peak = act.max()
    start_h = ts_idx[0].strftime('%H')
    ax.set_title(f'{pd.Timestamp(date).strftime("%a %b %d")} ({start_h}Z-+24h) '
                 f'Peak={peak:.0f} MW  MAE={mae:.1f} MW', fontsize=9)
    ax.set_ylabel('MW'); ax.set_xlim(0, 23)
    if i == 0:
        ax.legend(fontsize=7)

fig.suptitle('NYCA BTM Solar: Actual vs DA Forecast (20 days, 2024)\n'
             'Each panel covers the Eastern day (same window as scenario generation)',
             fontsize=13, y=1.005)
plt.tight_layout()
plt.show()

---
## 5. Per-Zone Time Series (Sample Zones)

Check the largest BTM solar zones individually: LONGIL, N.Y.C., CAPITL, HUD VL.

In [ ]:
# Full year overview for top 4 zones (2024)
top_zones = ['LONGIL', 'N.Y.C.', 'CAPITL', 'HUD VL']

fig, axes = plt.subplots(4, 1, figsize=(18, 16), sharex=True)
for ax, zone in zip(axes, top_zones):
    data_2024 = btm_actual.loc['2024', zone]
    ax.plot(data_2024.index, data_2024.values, color='darkorange', alpha=0.6, linewidth=0.3)
    # Rolling 7-day max
    daily_max = data_2024.resample('D').max()
    ax.plot(daily_max.index, daily_max.values, color='red', linewidth=1, label='Daily peak')
    ax.set_ylabel(f'{zone} (MW)')
    ax.legend(fontsize=8, loc='upper left')

axes[-1].set_xlabel('Date')
plt.suptitle('BTM Solar Estimated Actuals — Top 4 Zones (2024)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Zone shares of total BTM solar (2024)
zone_totals = btm_actual.loc['2024', ZONES].sum()
zone_pct = (zone_totals / zone_totals.sum() * 100).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(zone_pct)))
ax.barh(range(len(zone_pct)), zone_pct.values, color=colors)
ax.set_yticks(range(len(zone_pct)))
ax.set_yticklabels(zone_pct.index)
for i, (v, z) in enumerate(zip(zone_pct.values, zone_pct.index)):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=10)
ax.set_xlabel('Share of Total BTM Solar Energy (%)')
ax.set_title('BTM Solar Distribution by Zone (2024)')
plt.tight_layout()
plt.show()

---
## 6. PGScen GEMINI Scenario Generation

We treat BTM solar the same way as load: zonal data with actuals + day-ahead forecasts.
GEMINI fits the spatio-temporal correlation of forecast deviations and generates
Monte Carlo scenarios.

**Key adaptations for BTM solar:**

- `asset_type='wind'` (conditional ECDF marginals) — BTM solar has all-zero nighttime
  hours where GPD fitting would crash.
- `forecast_lead_time_in_hour=10` — NYISO DA forecast covers 00:00-23:00 Eastern.
- **Relative deviation training** — BTM solar capacity grows ~30%/year. Training on
  absolute deviations (MW) conflates capacity growth with forecast error, producing
  scenarios that are too narrow for the current year. We normalize by the concurrent
  forecast: `actual/forecast` and `forecast/forecast = 1`, so GEMINI trains on
  *relative* deviations. After generation, scenarios are rescaled to MW using the
  target day's forecast. This makes the learned distribution scale-invariant.

In [ ]:
from pgscen.engine import GeminiEngine
from pgscen.utils.data_utils import (
    split_actuals_hist_future, split_forecasts_hist_future
)
import time

LEAD_HOURS = 10
TRAIN_MONTHS = 12


def fix_issue_times(forecast_df, lead_hours=LEAD_HOURS):
    """Recompute Issue_time so that Issue_time + lead_hours = first Forecast_time."""
    fc = forecast_df.copy()
    new_issue = []
    for it, group in fc.groupby('Issue_time'):
        first_ft = group['Forecast_time'].min()
        new_it = first_ft - pd.Timedelta(hours=lead_hours)
        new_issue.extend([new_it] * len(group))
    fc['Issue_time'] = new_issue
    return fc


def add_solar_noise(act_df, fc_df, zones, eps=0.01):
    """Add tiny noise to break zero-degeneracy at nighttime hours."""
    act = act_df.copy(); fc = fc_df.copy()
    rng = np.random.RandomState(42)
    for z in zones:
        m = act[z] == 0; act.loc[m, z] = rng.uniform(0, eps, size=m.sum())
        m = fc[z] == 0;  fc.loc[m, z] = rng.uniform(0, eps, size=m.sum())
    return act, fc


def normalize_by_forecast(act_df, fc_df, zones, floor=1.0):
    """
    Normalize actuals and forecasts by dividing by the forecast value.
    
    After normalization, GEMINI trains on relative deviations:
      deviation = actual/forecast - forecast/forecast = actual/forecast - 1
    This makes the distribution scale-invariant across years with different
    installed BTM capacity.
    
    `floor` prevents division by zero at nighttime (default: 1 MW).
    """
    act_norm = act_df.copy(); fc_norm = fc_df.copy()
    fc_aligned = fc_df.set_index('Forecast_time').drop(columns='Issue_time')
    fc_aligned.index = pd.to_datetime(fc_aligned.index, utc=True)
    common = act_df.index.intersection(fc_aligned.index)
    for z in zones:
        ref = fc_aligned.loc[common, z].reindex(act_df.index).clip(lower=floor)
        act_norm[z] = act_df[z] / ref
        act_norm[z] = act_norm[z].fillna(act_df[z] / floor)
    for z in zones:
        fc_norm[z] = fc_df[z] / fc_df[z].clip(lower=floor)
    return act_norm, fc_norm


def run_btm_scenarios(btm_actual, btm_forecast_fixed, date, n_scenarios=1000,
                      asset_rho=0.05, time_rho=0.1, train_months=TRAIN_MONTHS):
    """
    Run PGScen GEMINI on BTM solar data for one date.
    
    1. Restrict training to last `train_months` months
    2. Normalize actuals & forecasts by forecast (relative deviations)
    3. Train GEMINI in relative space
    4. Generate relative scenarios, denormalize to MW
    """
    target_date = pd.Timestamp(date, tz='UTC')

    fc_day = btm_forecast_fixed[
        (btm_forecast_fixed['Forecast_time'] >= target_date) &
        (btm_forecast_fixed['Forecast_time'] < target_date + pd.Timedelta(hours=30))
    ]
    if len(fc_day) == 0:
        return None

    it_target = fc_day['Issue_time'].iloc[0]
    fc_block = btm_forecast_fixed[btm_forecast_fixed['Issue_time'] == it_target]
    if len(fc_block) != 24:
        return None

    scen_start = fc_block['Forecast_time'].min()
    scen_ts = pd.date_range(scen_start, periods=24, freq='h')
    zones = [c for c in btm_actual.columns if c != 'NYCA']

    # Restrict training window
    train_start = scen_ts[0] - pd.DateOffset(months=train_months)
    act_raw = btm_actual[zones].loc[train_start:]
    fc_raw = btm_forecast_fixed[['Issue_time', 'Forecast_time'] + zones]
    fc_raw = fc_raw[fc_raw['Forecast_time'] >= train_start].copy()

    try:
        # Normalize by forecast
        act_norm, fc_norm = normalize_by_forecast(act_raw, fc_raw, zones)

        act_h, _ = split_actuals_hist_future(act_norm, scen_ts, in_sample=False)
        fc_h, fc_f = split_forecasts_hist_future(fc_norm, scen_ts, in_sample=False)
        _, act_f_raw = split_actuals_hist_future(act_raw, scen_ts, in_sample=False)
        _, fc_f_raw = split_forecasts_hist_future(fc_raw, scen_ts, in_sample=False)

        if len(fc_f) < 24:
            return None

        act_h_noisy, fc_h_noisy = add_solar_noise(act_h, fc_h, zones)

        ge = GeminiEngine(act_h_noisy, fc_h_noisy, scen_ts[0],
                          asset_type='wind',
                          forecast_lead_time_in_hour=LEAD_HOURS)
        ge.fit(asset_rho, time_rho)
        ge.create_scenario(n_scenarios, fc_f)

        scen = ge.scenarios['wind']

        # Denormalize: relative -> MW
        fc_mw = fc_f_raw.set_index('Forecast_time').drop(columns='Issue_time').loc[scen_ts]

        zone_scens = {}
        fleet = np.zeros((n_scenarios, 24))
        for z in zones:
            z_cols = [(z, t) for t in scen_ts]
            if z_cols[0] in scen.columns:
                rel = scen[z_cols].values
                mw = rel * fc_mw[z].values[np.newaxis, :]
                mw = np.maximum(mw, 0)
                zone_scens[z] = mw
                fleet += mw

        return {
            'scen_ts': scen_ts, 'zones': zones, 'zone_scens': zone_scens,
            'fleet': fleet, 'act_f': act_f_raw, 'fc_f': fc_f_raw, 'engine': ge,
        }
    except Exception as e:
        print(f'  {date}: FAILED - {e}')
        return None


# Pre-compute fixed Issue_times
btm_forecast_fixed = fix_issue_times(btm_forecast)
print(f'Fixed forecast: {len(btm_forecast_fixed)} rows, {btm_forecast_fixed.Issue_time.nunique()} blocks')
print(f'Training: {TRAIN_MONTHS}-month window, relative deviations')
print('Scenario helper ready')

In [ ]:
# Quick test on one date
t0 = time.time()
test_result = run_btm_scenarios(btm_actual, btm_forecast_fixed, '2024-07-04')
if test_result:
    print(f'Success in {time.time()-t0:.1f}s')
    print(f'Fleet scenarios shape: {test_result["fleet"].shape}')
    print(f'Scenario timesteps: {test_result["scen_ts"][0]} -> {test_result["scen_ts"][-1]}')
    print(f'Zones: {test_result["zones"]}')
else:
    print('Failed!')

---
## 7. Scenario Visualization — NYCA Fleet, 20 Days

We generate 1000 scenarios for each of 20 days spread across 2024 and check
calibration: what fraction of hours has the actual outside the P5-P95 band?

Target: ~10% exceedance (well-calibrated).

In [ ]:
# Night hours (UTC) where BTM solar is always zero
NIGHT_HOURS_UTC = list(range(0, 4)) + list(range(24, 30))  # before sunrise in UTC

fig, axes = plt.subplots(10, 2, figsize=(18, 50))
n_out_fleet, n_tot_fleet = 0, 0
n_out_fleet99, n_tot_fleet99 = 0, 0

for i, date in enumerate(dates_2024):
    ax = axes.flatten()[i]
    r = run_btm_scenarios(btm_actual, btm_forecast_fixed, date)
    if r is None:
        ax.set_title(f'{date} - FAILED'); continue
    
    fleet = r['fleet']
    scen_ts = r['scen_ts']
    # Sum actuals and forecasts across zones
    afl = r['act_f'].loc[scen_ts][r['zones']].sum(axis=1).values
    fc_f_aligned = r['fc_f'].set_index('Forecast_time').drop(columns='Issue_time')
    ffl = fc_f_aligned.loc[scen_ts][r['zones']].sum(axis=1).values
    
    # Clamp scenarios to >= 0 (solar can't be negative)
    fleet = np.maximum(fleet, 0)
    
    # Determine daytime hours (where actual > 0 for at least some part of the year)
    night_mask = [h for h in range(24) if afl[h] == 0 and ffl[h] == 0]
    
    for j, (lo, hi) in enumerate([(1,99),(5,95),(10,90),(25,75)]):
        ax.fill_between(range(24),
                        np.percentile(fleet, lo, axis=0),
                        np.percentile(fleet, hi, axis=0),
                        alpha=0.5, color=['#fdebd0','#f9e79f','#f4d03f','#d4ac0d'][j])
    ax.plot(range(24), np.median(fleet, axis=0), color='darkorange',
            linewidth=2, label='P50')
    ax.plot(range(24), afl, color='red', linewidth=2, marker='o', ms=3, label='Actual')
    ax.plot(range(24), ffl, color='blue', linewidth=1.8, ls='--', label='Forecast')
    
    # Calibration (daytime only — skip hours where both actual and forecast are 0)
    p5 = np.percentile(fleet, 5, axis=0)
    p95 = np.percentile(fleet, 95, axis=0)
    p1 = np.percentile(fleet, 1, axis=0)
    p99 = np.percentile(fleet, 99, axis=0)
    
    out95, out99, day_hrs = 0, 0, 0
    for h in range(24):
        if h in night_mask:
            continue
        day_hrs += 1
        if afl[h] < p5[h] or afl[h] > p95[h]:
            out95 += 1
        if afl[h] < p1[h] or afl[h] > p99[h]:
            out99 += 1
    n_out_fleet += out95; n_tot_fleet += day_hrs
    n_out_fleet99 += out99; n_tot_fleet99 += day_hrs
    
    start_h = scen_ts[0].strftime('%H')
    ax.set_title(f'{pd.Timestamp(date).strftime("%a %b %d")} (start {start_h}Z) '
                 f'Peak={afl.max():.0f} MW  Out={out95}/{day_hrs}', fontsize=9)
    ax.set_ylabel('Fleet MW'); ax.set_xlim(0, 23); ax.set_ylim(bottom=0)
    if i == 0:
        ax.legend(fontsize=7)

pct_fleet = 100 * n_out_fleet / max(n_tot_fleet, 1)
pct_fleet99 = 100 * n_out_fleet99 / max(n_tot_fleet99, 1)
fig.suptitle(f'BTM Solar Fleet (11 zones) - 1000 Scenarios\n'
             f'Daytime P5-P95: {n_out_fleet}/{n_tot_fleet} ({pct_fleet:.1f}%, target ~10%)  |  '
             f'P1-P99: {n_out_fleet99}/{n_tot_fleet99} ({pct_fleet99:.1f}%, target ~2%)',
             fontsize=13, y=1.005)
plt.tight_layout()
plt.show()

print(f'BTM Solar fleet (daytime): P5-P95={pct_fleet:.1f}%, P1-P99={pct_fleet99:.1f}%')

---
## 8. Per-Zone Scenarios — LONGIL (Largest Zone)

Long Island has the most BTM solar in NY. We check per-zone calibration.

In [ ]:
REF_ZONE = 'LONGIL'

fig, axes = plt.subplots(10, 2, figsize=(18, 50))
n_out_z, n_tot_z = 0, 0
n_out_z99, n_tot_z99 = 0, 0

for i, date in enumerate(dates_2024):
    ax = axes.flatten()[i]
    r = run_btm_scenarios(btm_actual, btm_forecast_fixed, date)
    if r is None or REF_ZONE not in r['zone_scens']:
        ax.set_title(f'{date} - FAILED'); continue
    
    zs = np.maximum(r['zone_scens'][REF_ZONE], 0)
    scen_ts = r['scen_ts']
    az = r['act_f'].loc[scen_ts, REF_ZONE].values
    fz = r['fc_f'].set_index('Forecast_time')[REF_ZONE].loc[scen_ts].values
    
    night_mask = [h for h in range(24) if az[h] == 0 and fz[h] == 0]
    
    for j, (lo, hi) in enumerate([(1,99),(5,95),(10,90),(25,75)]):
        ax.fill_between(range(24),
                        np.percentile(zs, lo, axis=0),
                        np.percentile(zs, hi, axis=0),
                        alpha=0.5, color=['#fdebd0','#f9e79f','#f4d03f','#d4ac0d'][j])
    ax.plot(range(24), np.median(zs, axis=0), color='darkorange',
            linewidth=2, label='P50')
    ax.plot(range(24), az, color='red', linewidth=2, marker='o', ms=3, label='Actual')
    ax.plot(range(24), fz, color='blue', linewidth=1.8, ls='--', label='Forecast')
    
    p5 = np.percentile(zs, 5, axis=0); p95 = np.percentile(zs, 95, axis=0)
    p1 = np.percentile(zs, 1, axis=0); p99 = np.percentile(zs, 99, axis=0)
    out95, out99, day_hrs = 0, 0, 0
    for h in range(24):
        if h in night_mask:
            continue
        day_hrs += 1
        if az[h] < p5[h] or az[h] > p95[h]:
            out95 += 1
        if az[h] < p1[h] or az[h] > p99[h]:
            out99 += 1
    n_out_z += out95; n_tot_z += day_hrs
    n_out_z99 += out99; n_tot_z99 += day_hrs
    
    ax.set_title(f'{pd.Timestamp(date).strftime("%a %b %d")} '
                 f'Peak={az.max():.0f}  Out={out95}/{day_hrs}', fontsize=9)
    ax.set_ylabel('MW'); ax.set_xlim(0, 23); ax.set_ylim(bottom=0)
    if i == 0:
        ax.legend(fontsize=7)

pct_z = 100 * n_out_z / max(n_tot_z, 1)
pct_z99 = 100 * n_out_z99 / max(n_tot_z99, 1)
fig.suptitle(f'{REF_ZONE} BTM Solar - 1000 Scenarios\n'
             f'Daytime P5-P95: {n_out_z}/{n_tot_z} ({pct_z:.1f}%)  |  '
             f'P1-P99: {n_out_z99}/{n_tot_z99} ({pct_z99:.1f}%)',
             fontsize=13, y=1.005)
plt.tight_layout()
plt.show()

print(f'{REF_ZONE} (daytime): P5-P95={pct_z:.1f}%, P1-P99={pct_z99:.1f}%')

---
## 9. Per-Zone Calibration Summary — All Zones

Run scenarios for 10 dates and check P5-P95 exceedance for every zone.

In [ ]:
test_dates = dates_2024[::2]  # 10 dates

zone_stats = {z: [0, 0] for z in ZONES}
fleet_stats = [0, 0]

for date in test_dates:
    r = run_btm_scenarios(btm_actual, btm_forecast_fixed, date)
    if r is None:
        continue
    
    scen_ts = r['scen_ts']
    afl_full = r['act_f'].loc[scen_ts][r['zones']].sum(axis=1).values
    ffl_full = r['fc_f'].set_index('Forecast_time').drop(columns='Issue_time').loc[scen_ts][r['zones']].sum(axis=1).values
    
    # Per-zone
    for z in r['zones']:
        if z not in r['zone_scens']:
            continue
        zs = np.maximum(r['zone_scens'][z], 0)
        az = r['act_f'].loc[scen_ts, z].values
        fz = r['fc_f'].set_index('Forecast_time')[z].loc[scen_ts].values
        for h in range(24):
            if az[h] == 0 and fz[h] == 0:
                continue
            p5 = np.percentile(zs[:, h], 5)
            p95 = np.percentile(zs[:, h], 95)
            if az[h] < p5 or az[h] > p95:
                zone_stats[z][0] += 1
            zone_stats[z][1] += 1
    
    # Fleet
    fleet = np.maximum(r['fleet'], 0)
    for h in range(24):
        if afl_full[h] == 0 and ffl_full[h] == 0:
            continue
        p5 = np.percentile(fleet[:, h], 5)
        p95 = np.percentile(fleet[:, h], 95)
        if afl_full[h] < p5 or afl_full[h] > p95:
            fleet_stats[0] += 1
        fleet_stats[1] += 1

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Per-zone
ax = axes[0]
z_pcts = {z: 100*s[0]/s[1] if s[1] > 0 else 0 for z, s in zone_stats.items()}
z_sorted = sorted(z_pcts.items(), key=lambda x: x[1])
names = [x[0] for x in z_sorted]
vals = [x[1] for x in z_sorted]
colors_z = ['green' if 5 <= v <= 15 else 'orange' if v <= 25 else 'red' for v in vals]
ax.barh(range(len(names)), vals, color=colors_z, alpha=0.7)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.axvline(10, color='red', ls='--', label='Target 10%')
ax.axvspan(5, 15, alpha=0.1, color='green', label='Acceptable')
ax.set_xlabel('% outside P5-P95 (daytime)')
ax.set_title('BTM Solar \u2014 Per-Zone Calibration')
ax.legend(fontsize=8)

# Summary bar
ax = axes[1]
labels = ['Fleet\n(all zones)', REF_ZONE]
fleet_pct = 100 * fleet_stats[0] / max(fleet_stats[1], 1)
ref_pct = z_pcts.get(REF_ZONE, 0)
vals_bar = [fleet_pct, ref_pct]
colors_bar = ['darkorange', 'gold']
bars = ax.bar(labels, vals_bar, color=colors_bar, alpha=0.7, edgecolor='gray')
ax.axhline(10, color='red', ls='--', linewidth=2, label='Target (10%)')
ax.axhspan(5, 15, alpha=0.1, color='green', label='Acceptable')
for bar, val in zip(bars, vals_bar):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('% hours outside P5-P95')
ax.set_title('BTM Solar Calibration Summary')
ax.legend(fontsize=9)
ax.set_ylim(0, max(max(vals_bar), 15) * 1.3)

plt.suptitle('BTM Solar Scenario Calibration \u2014 10 days across 2024 (daytime only)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

z_ok = sum(1 for v in z_pcts.values() if 3 <= v <= 20)
print(f'Zones well-calibrated: {z_ok}/{len(z_pcts)}')
print(f'Fleet: {fleet_pct:.1f}%')

---
## 10. GEMINI Spatio-Temporal Correlations

Visualize the correlation structure learned by GEMINI for BTM solar.

In [ ]:
# Use the July 4 result
r = run_btm_scenarios(btm_actual, btm_forecast_fixed, '2024-07-04')
if r:
    model = r['engine'].model
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Temporal
    h_cov = model.horizon_cov
    if h_cov is not None:
        d = np.sqrt(np.diag(h_cov.values))
        d[d == 0] = 1
        h_corr = h_cov.values / np.outer(d, d)
        sns.heatmap(h_corr, ax=axes[0], cmap='RdBu_r', center=0,
                    vmin=-1, vmax=1, square=True,
                    xticklabels=range(24), yticklabels=range(24))
        axes[0].set_title('Temporal Component\n(conditional correlation)')
        axes[0].set_xlabel('Hour'); axes[0].set_ylabel('Hour')
    
    # Spatial
    a_cov = model.asset_cov
    if a_cov is not None:
        d = np.sqrt(np.diag(a_cov.values))
        d[d == 0] = 1
        a_corr = a_cov.values / np.outer(d, d)
        sns.heatmap(a_corr, ax=axes[1], cmap='RdBu_r', center=0,
                    vmin=-1, vmax=1, square=True, annot=True, fmt='.2f',
                    xticklabels=a_cov.columns, yticklabels=a_cov.index)
        axes[1].set_title('Spatial Component\n(conditional correlation between zones)')
        axes[1].tick_params(axis='x', rotation=45)
    
    plt.suptitle('BTM Solar \u2014 GEMINI Correlations (July 4, 2024)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Scenario generation failed for July 4, 2024')

---
## 11. Summary

| Metric | Value |
|--------|-------|
| Data source | NYISO OASIS P-70A (actuals), P-70B (DA forecast) |
| Available from | November 2020 |
| Resolution | Hourly, by zone |
| Zones | 11 NYISO zones + NYCA system total |
| 2024 peak NYCA | ~2.5 GW |
| Scenario engine | GEMINI (same as load) |

The BTM solar data follows the same format as the NYISO load data and
integrates directly into the PGScen pipeline.